In [1]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False, json.loads(sys.argv[1])['scenario'])

# START

In [2]:
import sys
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import stepmodel, connectionscanmodel
from quetzal.io import excel
import geopandas as gpd
import shapely
import os
import numpy as np
import pandas as pd
if MANUAL:
    %matplotlib inline

from syspy.syspy_utils.data_visualization import bandwidth, width_series, color_series

In [3]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [4]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

In [5]:
INTRA_AREA_TO_DISTANCE_FACTOR = var[('step_distribution', 'intra_area_to_distance_factor')]

WALK_TIME_FACTOR = var[('mode_penalty', 'walk_time_factor')]
BIKE_TIME_FACTOR = var[('mode_penalty', 'bike_time_factor')]
CAR_TIME_FACTOR = var[('mode_penalty', 'car_time_factor')]

WAITING_TIME_FACTOR = var[('pt_penalty', 'waiting_time_factor')]
TRANSFER_TIME_FACTOR = var[('pt_penalty', 'transfer_time_factor')]
ACCESS_TIME_FACTOR = var[('pt_penalty', 'access_time_factor')]
TRANSFER_PENALTY = var[('pt_penalty', 'transfer_penalty')]

KM_FARES = var['fare_km'].to_dict()

CAR_KM_FARES = var['fare_car'].to_dict()
KM_FARES['car'] = CAR_KM_FARES['fuel_cost'] + CAR_KM_FARES['other_costs']

VALUE_OF_TIME_SEC = var[('value_of_time', 'value_of_time_h')]/3600

In [6]:
if var['run_mode']['mode'] != 'timetables':
    sm = stepmodel.read_zippedpickles(mpath + r'generation_distribution')
else:
    sm = connectionscanmodel.read_zippedpickles(mpath + r'generation_distribution')

zone_to_road: 100%|██████████| 22/22 [00:02<00:00, 10.32it/s]    


## MaP

In [7]:
sm.los = sm.los[sm.los["route_type"]!="walk"]
sm.segments = ["volume"]

In [8]:
sm.los.loc[(sm.los['origin'] == 'zone_05') & (sm.los['destination'] == 'zone_04')][['route_types'] + [x for x in sm.los.columns if 'time' in x] + ['length', 'fare']]

,route_types,time,gtime,time_link_path,access_time,footpath_time,in_vehicle_time,waiting_time,boarding_time,length,fare
339,"(car,)",3323.819209,9977.144873,NaN,NaN,NaN,NaN,NaN,NaN,44056.536505,9.362014
4895,"(bike,)",9977.144873,9249.149228,NaN,NaN,NaN,NaN,NaN,NaN,43105.846878,0.000000
9451,"(bus,)",20197.273841,23696.772595,9019.498754,1010.307214,2866.966627,5520.0,10800.0,0,NaN,6.864040
38493,"(bus,)",21456.726646,24971.128154,8614.401508,1256.671291,4300.055355,5100.0,10800.0,0,NaN,6.884764
40612,"(bus,)",25414.661561,29587.102656,10472.441094,1010.307214,8504.354347,6300.0,9600.0,0,NaN,7.799098


In [9]:
### Compute zone-zone distances

sm.zones['area'] = sm.zones.area ## m²

distances = sm.zones.reset_index()[['index', 'lat_centroid', 'lng_centroid', 'area']]
distances['geometry'] = gpd.points_from_xy(distances['lng_centroid'], distances['lat_centroid'])
distances = distances.set_geometry('geometry').set_crs('EPSG:4326').to_crs('EPSG:2154').drop(columns=['lng_centroid', 'lat_centroid'])
distances = distances.merge(distances, how='cross', suffixes = ['_from', '_to'])
distances['distance'] = (distances['area_from'] ** 0.5) * INTRA_AREA_TO_DISTANCE_FACTOR
distances.loc[distances['index_from'] != distances['index_to'], 'distance'] = distances.apply(lambda x: shapely.distance(x['geometry_from'], x['geometry_to']), 1)
distances = distances[['index_from', 'index_to', 'distance']].rename(columns={'index_from': 'origin', 'index_to': 'destination'})

In [10]:
sm.volumes

,origin,destination,volume
0,zone_00,zone_00,510.152397
1,zone_00,zone_01,545.589559
2,zone_00,zone_02,33.679766
3,zone_00,zone_03,34.005669
4,zone_00,zone_04,59.787849
...,...,...,...
4619,zone_67,zone_63,185.196626
4620,zone_67,zone_64,538.109512
4621,zone_67,zone_65,483.048403
4622,zone_67,zone_66,524.909599


In [11]:
### Share of walk among volumes:
share_not_walking = {
    1000: 0.12,
    2000: 0.41,
    3000: 0.87,
    4000: 0.97,
    5000: 0.98, 
    np.inf: 1
}

def apply_dict(x, dict):
    for v_ in dict.keys():
        if x <= v_:
            return dict[v_]

sm.volumes = sm.volumes.merge(distances, on=['origin', 'destination'], how='left')
sm.walk_volumes = sm.volumes.copy()
sm.walk_volumes['volume'] *= (1-sm.volumes['distance'].apply(lambda x: apply_dict(x, share_not_walking)))
sm.volumes['volume'] *= sm.volumes['distance'].apply(lambda x: apply_dict(x, share_not_walking))
sm.volumes.drop(columns='distance', inplace=True)
# sm.walk_volumes.drop(columns='distance', inplace=True)

In [12]:
# TODO: gérer la MAP pour les intrazonaux des zones très grandes -> mini 25% de MAP imposée ?

In [13]:
sm.walk_volumes[sm.walk_volumes.origin==sm.walk_volumes.destination]

,origin,destination,volume,distance
0,zone_00,zone_00,0.000000,29826.229948
69,zone_01,zone_01,0.000000,25434.582642
138,zone_02,zone_02,0.000000,15161.315131
207,zone_03,zone_03,0.000000,24408.062517
276,zone_04,zone_04,0.000000,25169.608032
...,...,...,...,...
4347,zone_63,zone_63,1495.482665,1326.223999
4416,zone_64,zone_64,2413.866490,1285.117223
4485,zone_65,zone_65,577.950948,1041.077031
4554,zone_66,zone_66,1032.996314,1273.694369


## Configure Logit scales

In [14]:
var['preparation_logit'].to_dict()

{'time': -0.0005,
 'price': -0.2,
 'transfers': -0.5,
 'mode': 1,
 'pt_mode': 0.8,
 'pt_path': 0.5}

In [15]:
kwargs = var['preparation_logit'].to_dict()

sm.preparation_logit(
    **kwargs, segments=['volume'],
    route_types=['root', 'car', 'bike', 'pt'] + list(sm.links["route_type"].unique())
)

In [16]:
sm.utility_values.T

value,time,price,ntransfers,mode_utility
segment,,,,
root,-0.0005,-0.2,-0.5,1.0
volume,-0.0005,-0.2,-0.5,1.0


In [17]:
sm.logit_scales.T

route_type,root,car,bike,pt,bus,rail,tramway
segment,,,,,,,
root,1.0,0.0,0.5,0.8,0.5,0.5,0.5
volume,1.0,0.0,0.5,0.8,0.5,0.5,0.5


In [18]:
sm.mode_nests.loc['bike'] = 'root'
sm.mode_nests

segment,root,volume
route_type,,
root,NaN,NaN
car,root,root
bike,root,root
pt,root,root
bus,pt,pt
rail,pt,pt
tramway,pt,pt


In [19]:
# TODO: ajouter une valeur par défaut pour les paramètres de mode TC manquants ?
for mode in set(sm.mode_utility.index) - {"root", "pt"}:
    sm.mode_utility.loc[mode] = var['mode_utility'][mode]

sm.mode_utility

segment,root,volume
route_type,,
root,0,0
car,1,1
bike,0,0
pt,0,0
bus,0,0
rail,0,0
tramway,0,0


## Utility

In [20]:

# TODO Louison: (temps de transfer? = temps d'attente pas lié au premier moyen de transport utilisé?)
# temps de rabattement : access_time
# temps de correspondance : waiting_time
# temps d'accès : footpath_time
### et finalement : time = in_vehicle_time + access_time + footpath_time + waiting_time

# TODO : BOUCLE à crée avec les cellules dessous, à mettre au propre dans des fonctions avant

Principe: affectation par tranche et mise à jour des LOS
on découpe la demande en 10 (paramétrable) tranches et on fait 10 itérations:

- a. Logit
- b. on fait l’affectation TC et on met à jour les LOS TC
- c. on fait l’affectation CAR (AON) et on met à jour les LOS CAR
- d. on fusionne tous les LOS mis à jour et on passe à la tranche suivante

Les tranches sont de taille différente (plus grosses au début)

## A. RUN LOGIT

In [21]:
## TODO: à mettre dans networks
mode_capacities = {
    'tramway': 500,
    'bus': 80,
    'rail': 800
}

sm.links['mode_capacity'] = sm.links['route_type'].map(mode_capacities)
sm.links['capacity'] = 3600 * sm.links['mode_capacity'] / sm.links['headway']

# B. ASSIGNMENT

In [22]:
congestion = True

def _build_car_los(car_los):
    car_los['penalties'] = 0
    car_los['ntransfers'] = 0
    car_los['parking_fare'] = car_los['destination'].map(sm.zones['parking_fare'].to_dict())
    car_los['flat_fare'] = 0
    car_los['km_fare'] = car_los['length'] * KM_FARES['car'] / 1000 
    car_los['fare'] = car_los['length'] * KM_FARES['car'] / 1000 + car_los['parking_fare']
    car_los['time'] *= CAR_TIME_FACTOR
    car_los['time'] += car_los['penalties']
    car_los['route_types'] = [('car',)] * len(car_los)
    car_los['route_type'] = 'car'
    # car_los['time'] = car_los['gtime']
    car_los['gtime'] = car_los["time"] + car_los['in_vehicle_length'] / 1000 * KM_FARES['car'] / VALUE_OF_TIME_SEC
    return car_los

def get_boarding_bpr(row):
        p = 1
        delta = 0.5
        # boarding_ctime = ((row['boardings'] / max(row['capacity'] - row['msa_load'], 1e-5)) ** p) * (row['bpr'] - 1) * t0
        boarding_bpr = delta * (row['boardings'] / max(row['capacity'] - row['msa_load'], 1e-5)) ** p
        return boarding_bpr

def _apply_los_time_factors(los):
    pt_mask = los['route_type'].isin(['rail', 'tramway', 'bus'])
    car_mask = (los['route_type'] == 'car')
    bike_mask = (los['route_type'] == 'bike')
    walk_mask = (los['route_type'] == 'walk')

    los.loc[pt_mask, 'time'] = los.loc[pt_mask, 'in_vehicle_time'].fillna(0) +\
        los.loc[pt_mask, 'access_time'].fillna(0)*ACCESS_TIME_FACTOR +\
        los.loc[pt_mask, 'footpath_time'].fillna(0)*ACCESS_TIME_FACTOR +\
        los.loc[pt_mask, 'waiting_time'].fillna(0)*WAITING_TIME_FACTOR +\
        los.loc[pt_mask, 'ntransfers']*TRANSFER_PENALTY+\
        los.loc[pt_mask, 'in_vehicle_congestion_penalty']+\
        los.loc[pt_mask, 'boarding_congestion_penalty']

    los.loc[car_mask, 'time'] = los.loc[car_mask, 'time']*CAR_TIME_FACTOR

    los.loc[bike_mask, 'time'] = los.loc[bike_mask, 'time'] * BIKE_TIME_FACTOR + 0.01 * los.loc[bike_mask, 'length'] ** 1.5
    los.loc[walk_mask, 'time'] = los.loc[walk_mask, 'time'] * WALK_TIME_FACTOR
    return los

In [23]:
eta = 0.7
y = [1 / (1+(i-1) * eta) for i in range(1,10)]
y
[(1/2)**((i-1)/2) for i in range(1,10)]

[1.0,
 0.7071067811865476,
 0.5,
 0.3535533905932738,
 0.25,
 0.1767766952966369,
 0.125,
 0.08838834764831845,
 0.0625]

,highway,speed,lanes,name,oneway,surface,a,b,bike_index,length,...,short_leg_speed,long_leg_speed,flow,auxiliary_flow,"(volume, flow)","(volume, auxiliary_flow)",base_flow,jam_time,sparse_a,sparse_b
ztr_0,NaN,3.5,NaN,NaN,NaN,NaN,zone_00,rnode_202192,NaN,824.837958,...,3.5,3.5,0,0,0,0,0,848.404756,58184,18875
ztr_1,NaN,3.5,NaN,NaN,NaN,NaN,zone_01,rnode_285361,NaN,324.070298,...,3.5,3.5,0,0,0,0,0,333.329449,58185,34132
ztr_2,NaN,3.5,NaN,NaN,NaN,NaN,zone_02,rnode_54704,NaN,412.284857,...,3.5,3.5,0,0,0,0,0,424.064424,58186,49767
ztr_3,NaN,3.5,NaN,NaN,NaN,NaN,zone_03,rnode_333952,NaN,134.861266,...,3.5,3.5,0,0,0,0,0,138.714445,58187,43161
ztr_4,NaN,3.5,NaN,NaN,NaN,NaN,zone_04,rnode_349068,NaN,409.395020,...,3.5,3.5,0,0,0,0,0,421.092021,58188,45968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ztr_675,NaN,3.5,NaN,NaN,NaN,NaN,rnode_312669,zone_63,NaN,117.809663,...,3.5,3.5,0,0,0,0,0,121.175653,39194,58247
ztr_676,NaN,3.5,NaN,NaN,NaN,NaN,rnode_347339,zone_64,NaN,188.571227,...,3.5,3.5,0,0,0,0,0,193.958976,45663,58248
ztr_677,NaN,3.5,NaN,NaN,NaN,NaN,rnode_123874,zone_65,NaN,48.940364,...,3.5,3.5,0,0,0,0,0,50.338660,4424,58249
ztr_678,NaN,3.5,NaN,NaN,NaN,NaN,rnode_16203,zone_66,NaN,86.872836,...,3.5,3.5,0,0,0,0,0,89.354917,11409,58250


In [ ]:
# compute car los
from quetzal.engine.road_pathfinder import (
    init_network,
    init_volumes,
    msa_roadpathfinder,
    get_car_los_time,
    to_polars
)
from quetzal.engine.msa_utils import jam_time

phi = 0.5

## PREPARATION
# base vdf
free_flow = 'time'
default_bpr = 'time * (1 + {alpha} * (flow/capacity)**{beta})'.format(alpha=0.15, beta=4)
od_set=None
method="msa"
segments = sm.segments
time_column = "time"
access_time = "time"
vdf={'default_bpr': default_bpr, 'free_flow': free_flow}
network = init_network(sm, method, segments, time_column, access_time)
network["capacity"] = network["capacity"].fillna(1)

car = sm.copy()
car.car_los = car.los.loc[car.los['route_type'] == 'car']
car.volumes = car.los.groupby(["origin", "destination"], as_index=False)[["volume"]].sum()

### ROAD ASSIGNMENT - 1ère itération
volumes = car.volumes
loaded_links, car_los, rel_gap = msa_roadpathfinder(
    network,
    volumes,
    segments=segments,
    method=method,
    time_col=time_column,
    maxiters=1,
    num_cores=1,
    return_car_los=True
)


### MAJ CAR_LOS
time_dict = loaded_links['jam_time'].to_dict()
car_los = get_car_los_time(car_los, time_dict)
car.car_los = car_los
car.analysis_car_los()
car.analysis_car_length()
car.car_los = _build_car_los(car.car_los)
loaded_links_bkp = loaded_links

### ITERATIONS SUIVANTES
volumes = car.car_los.groupby[]# TODO
loaded_links, car_los, rel_gap = msa_roadpathfinder(
    network,
    volumes,
    segments=segments,
    method=method,
    time_col=time_column,
    maxiters=1,
    num_cores=1,
    return_car_los=True
)

# auxiliary_flow est l’ancienne itération
loaded_links["flow"] = (1 - phi) * loaded_links["flow"] + (phi * loaded_links_bkp['flow']) 
loaded_links['jam_time'] = jam_time(to_polars(loaded_links), vdf, 'flow', time_col=time_column)

### MAJ CAR_LOS
time_dict = loaded_links['jam_time'].to_dict()
car_los = get_car_los_time(car_los, time_dict)
car.car_los = car_los
car.analysis_car_los()
car.analysis_car_length()
car.car_los = _build_car_los(car.car_los)
loaded_links_bkp = loaded_links_bkp


KeyError: 'auxiliary_flow'

In [51]:
car.car_los.head(2)

,origin,destination,path,segment,edge_path,time,gtime,node_path,link_path,ntlegs,...,in_vehicle_length,length,penalties,ntransfers,parking_fare,flat_fare,km_fare,fare,route_types,route_type
0,zone_00,zone_00,[zone_00],volume,[],0.000000,0.000000,[],[],[],...,0.000000,0.000000,0,0,0.0,0,0.000000,0.000000,"(car,)",car
1,zone_00,zone_01,"[zone_00, rnode_202192, rnode_80467, rnode_842...",volume,"[(zone_00, rnode_202192), (rnode_202192, rnode...",3569.704629,8789.021904,"[rnode_202192, rnode_80467, rnode_84202, rnode...","[rlink_279535, rlink_280404, rlink_280421, rli...","[(zone_00, rnode_202192), (rnode_285361, zone_...",...,34113.184802,35262.093058,0,0,0.0,0,7.493195,7.493195,"(car,)",car


In [23]:
sm.los["boarding_congestion_penalty"] = 0
sm.los["in_vehicle_congestion_penalty"] = 0
sm.los['price'] = sm.los['fare']
sm.los = _apply_los_time_factors(sm.los)
sm.zone_to_road["length"] = sm.zone_to_road["geometry"].apply(lambda g: g.length)  # TODO Louison: à mettre dans pathfinder

if congestion:

    sm_ = sm.copy()  # create copy for loop and los update

    n_iter = 10
    i = 0
    alpha = 2
    beta = 10
    eta = 0.7
    link_capacity_ratio = 0.9

    while i < n_iter:
        i += 1
        print(i)
        # phi = 1 / (1+(i-1) * eta)
        phi = (1/2)**((i-1)/2)

        # Calculate the utilities and the probabilities
        sm.analysis_mode_utility(how='mean')
        sm.initialize_logit()
        sm.step_logit()
        sm.compute_los_volume()

        temp = pd.concat([sm.los.groupby('route_type')['volume'].sum(), pd.Series({"walk": sm.walk_volumes["volume"].sum()})])
        print(100 * temp / temp.sum())

        # CAR
        car = sm.copy()
        car.car_los = car.los.loc[car.los['route_type'] == 'car']
        car.volumes = car.los.groupby(["origin", "destination"], as_index=False)[["volume"]].sum()

        # TODO Louison: ici on voudrait garder uniquement le réseau routier "car", mais on ne peut pas car les zone_to_road vont aussi vers le réseau vélo.
        # il faudrait donc ajouter des attributs car/bike aux zone_to_road également pour filtrer les réseau
        # en attendant, je garde le réseau vélo avec un capacité à 1 pour contourner le problème
        # car.road_links = car.road_links.loc[car.road_links["car"]]
        car.road_links["capacity"] = car.road_links["capacity"].fillna(1)

        # TODO: build car los aligned with PT MSA algorithm
        car.step_road_pathfinder(method="bfw", segments=["volume"], tolerance=5, maxiters=10, num_cores=4)
        car.analysis_car_los()
        car.analysis_car_time()
        car.analysis_car_length()
        car.car_los = _build_car_los(car.car_los)

        # PT    
        pt = sm.copy()
        pt.pt_los = pt.los[~(pt.los['route_type'].isin(['walk', 'car', 'bike']))]

        # PT ASSIGNMENT
        pt.step_pt_assignment(
            on_road_links=False, boardings=True, alightings=True, transfers=True
        )

        ## Congestion TC 
        ## 1. en ligne (links): ctime = time * (1 + alpha * (volume / capacity)**beta)
        ##      (Cela correspond en fait à un temps généralisé)
        ## 2. aux arrêts : boarding_ctime = waiting_time * (1 + delta * (boardings / remaining_capacity)**p)
        congested_links = pt.links
        if i == 1:
            congested_links['msa_load'] = congested_links['load']
            congested_links['bpr'] = 1 + alpha * ((congested_links['msa_load'] / (link_capacity_ratio*congested_links['capacity'])) ** beta)
            congested_links['ctime'] = congested_links['time'] * congested_links['bpr']
            congested_links['boarding_ctime'] = congested_links["headway"]/2 * (1 + congested_links.apply(get_boarding_bpr, axis=1))
        else:
            congested_links['msa_load'] = msa_bkp
            congested_links['msa_load'] = congested_links['load'] * phi + congested_links['msa_load'] * (1-phi)
            congested_links['bpr'] = 1 + alpha * ((congested_links['msa_load'] / (link_capacity_ratio*congested_links['capacity'])) ** beta)
            congested_links['ctime'] = congested_links['time'] * congested_links['bpr']
            congested_links['boarding_ctime'] = congested_links["headway"]/2 * (1 + congested_links.apply(get_boarding_bpr, axis=1))

            rel_gap = (
                np.sum(congested_links['load'] * congested_links['ctime']) - np.sum(msa_bkp * congested_links['ctime'])
                ) / np.sum(congested_links['load'] * congested_links['ctime'])
            rel_gap_2 = ((
                np.sum((congested_links['load'] - msa_bkp)**2)
                ) / np.sum(msa_bkp**2)) ** 0.5
            print(rel_gap, rel_gap_2)
            if abs(rel_gap) < 5*1e-4 and rel_gap_2 < 0.05:
                break

        msa_bkp = congested_links['msa_load']

        ## UPDATE LOS
        d = (congested_links['ctime'] - congested_links['time']).to_dict()
        pt.pt_los['in_vehicle_congestion_penalty'] = pt.pt_los['link_path'].apply(
            lambda l: 0.5 * sum([d.get(t, 0) for t in l]) if isinstance(l, list) else 0
        )
        d_boarding = (congested_links['boarding_ctime'] - congested_links["headway"]/2).to_dict()
        pt.pt_los['boarding_congestion_penalty'] = pt.pt_los['boarding_links'].apply(
            lambda l: 5 * sum([d_boarding.get(t, 0) for t in l]) if isinstance(l, list) else 0
        )

        pt.pt_los['time'] = pt.pt_los[
            ['access_time', 'footpath_time', 'in_vehicle_time', 'waiting_time', "boarding_time", 'in_vehicle_congestion_penalty', 'boarding_congestion_penalty']].T.sum()

        ## REBUILD LOS
        sm.los = pd.concat(
            [
                car.car_los,
                pt.pt_los,
                sm_.los.loc[sm_.los['route_type'] == 'bike']
            ]
        ).reset_index(drop=True)
        print("len sm.los: ", len(sm.los))
        sm.los['price'] = sm.los['fare']
        sm.los = _apply_los_time_factors(sm.los)
        
        

1


100%|██████████| 10/10 [00:00<00:00, 17.82it/s]


bike        2.516884
bus         2.066040
car        87.260568
rail        0.004004
tramway     1.890220
walk        6.262285
dtype: float64


volume: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]


len sm.los:  44197
2


100%|██████████| 10/10 [00:00<00:00, 31.10it/s]


... (2 outputs left out)

-0.010012605105182464 0.4309158194130204
len sm.los:  44197
3


100%|██████████| 10/10 [00:00<00:00, 31.48it/s]


bike        1.135301
bus         2.177978
car        88.182215
rail        0.005977
tramway     2.236244
walk        6.262285
dtype: float64


KeyboardInterrupt: 

In [24]:
congested_links.loc[congested_links.route_type=="tramway"][["route_id", "capacity", "headway", "bpr", "msa_load", "time", "ctime", "bpr", "boarding_ctime"]].sort_values("headway")

,route_id,capacity,headway,bpr,msa_load,time,ctime,bpr,boarding_ctime
index,,,,,,,,,
link_1_1059,F_TBM_0,3000.0,600,1.000464,1169.213620,105.0,105.048699,1.000464,347.098309
link_1_1060,F_TBM_0,3000.0,600,1.000464,1169.213620,110.0,110.051018,1.000464,300.000000
link_1_1061,F_TBM_0,3000.0,600,1.000000,555.194124,115.0,115.000031,1.000000,307.822600
link_1_1062,F_TBM_0,3000.0,600,1.000000,566.426557,85.0,85.000028,1.000000,300.797599
link_1_1063,F_TBM_0,3000.0,600,1.000000,566.426557,66.0,66.000022,1.000000,300.000000
...,...,...,...,...,...,...,...,...,...
link_1_4667,B_TBM_0,125.0,14400,1.000000,0.000000,77.0,77.000000,1.000000,7200.000000
link_1_4668,B_TBM_0,125.0,14400,1.000000,0.000000,117.0,117.000000,1.000000,7200.000000
link_1_4669,B_TBM_0,125.0,14400,1.000000,0.000000,71.0,71.000000,1.000000,7200.000000


In [25]:
cols = [c for c in sm.los.columns if "time" in c or "penalty" in c]
sm.los[sm.los["link_path"].apply(lambda p: "link_1_4423" in p)][cols].applymap(int)

,time,gtime,time_link_path,access_time,footpath_time,in_vehicle_time,waiting_time,boarding_time,boarding_congestion_penalty,in_vehicle_congestion_penalty
5410,18527,15692,5561,2153,5264,3778,2713,0,1893,768
5411,18493,15638,5779,2189,4956,3996,2713,0,1893,789
6409,498441,32577,12506,951,4406,8160,14713,0,460862,790
6410,502398,37111,14163,2138,6096,9240,14713,0,460862,790
6411,473663,12125,9166,647,558,6614,1753,0,461820,793
...,...,...,...,...,...,...,...,...,...,...
39218,615738,34695,17427,1551,4904,8366,10813,0,581999,897
39234,3319790,28482,13441,281,4846,5498,9913,0,3292799,295
39503,500098,22252,14047,1516,1815,6514,4873,0,480346,796
39504,533250,54081,15172,1574,26702,6918,10633,0,479511,795


### Bike

In [ ]:
# BIKE not needed in the loop as congestion does not impact bike
# bike = sm.copy()
# bike.car_los = bike.los.loc[bike.los['route_type'] == 'bike']
# bike.step_car_assignment()
# bike.road_links = bike.road_links.drop(columns=[('volume', 'car')]).rename(columns={('all', 'car'): 'volume_bike'})

volume: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]


### Compute modal share

In [26]:
temp = pd.concat([pt.los.groupby('route_type')['volume'].sum(), pd.Series({"walk": sm.walk_volumes["volume"].sum()})])
100 * temp / temp.sum()

bike       10.234371
bus         2.380066
car        78.982556
rail        0.006831
tramway     2.133890
walk        6.262285
dtype: float64

# EXPORT

In [75]:
car.road_links['width_cv'] = width_series(car.road_links['volume_car'])
car.road_links['color_cv'] = color_series(car.road_links['volume_car'], colors=["#210040", "#CFFF20"])
car.road_links['width_bv'] = width_series(car.road_links['volume_bike'])
car.road_links['color_bv'] = color_series(car.road_links['volume_bike'], colors=["#030F0D", "#6AE1FF"])
car.road_links['width_congestion'] = width_series(car.road_links['congestion'])
car.road_links['color_congestion'] = color_series(car.road_links['congestion'], colors=["#FFF7F7", "#B81313"])

pt.links['width_vol'] = width_series(pt.links['load'])
pt.links['color_vol'] = color_series(pt.links['load'], colors=["#210040", "#CFFF20"])
pt.links['width_congestion'] = width_series(pt.links['congestion'])
pt.links['color_congestion'] = color_series(pt.links['width_congestion'], colors=["#FFF7F7", "#B81313"])

In [76]:
car.road_links.drop(columns='index').to_file('../../scenarios/{}/outputs/loaded_road_links.geojson'.format(SCENARIO))
pt.links.to_file('../../scenarios/{}/outputs/pt_loaded_links.geojson'.format(SCENARIO))

In [77]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

In [ ]:
car.car_los = car.los.loc[car.los['route_type'] == 'car']
car.volumes

### bike = sm.copy()
### bike.car_los = bike.los.loc[bike.los['route_type'] == 'bike']
### bike.step_car_assignment()

,origin,destination,volume
0,zone_00,zone_00,519.720823
1,zone_00,zone_01,556.224143
2,zone_00,zone_02,34.601580
3,zone_00,zone_03,35.038249
4,zone_00,zone_04,61.429419
...,...,...,...
4619,zone_67,zone_63,194.918432
4620,zone_67,zone_64,666.183670
4621,zone_67,zone_65,301.423003
4622,zone_67,zone_66,67.116162


In [ ]:
sm.road_links["capacity"] = sm.road_links['lanes']*sm.road_links['speed']/(sm.road_links['length']/1000) ### TODO Louison : à partir de vitesse max et lanes
sm.zone_to_road["length"] = sm.zone_to_road.geometry.apply(lambda g: g.length)
car = sm.copy()
CAR_OCCUPANCY_RATE = 1.5
# CARS
car_los = sm.los.loc[sm.los['pathfinder_session']=='car_aon']
car_los['volume'] *= 1 / CAR_OCCUPANCY_RATE
car.volumes = car_los[["origin", "destination", "volume"]]

# car.step_road_pathfinder(method='aon', segments=["volume"])
car.step_road_pathfinder(method='aon', segments=["volume"], tolerance=10) # pour les itérations intermédiaires, pas besoin d’un relgap à 0.01%

# TODO Louison: remplacer roadpathfinder MSA par un simple AON + mise à jour des temps sur les road_links à partir des courbes BPR T0*(1+\a * (T/C)**\b)

In [ ]:
sm.pt_los = sm.los[~(sm.los['route_type'].isin(['walk', 'car', 'bike']))]

# PT ASSIGNMENT
sm.step_pt_assignment(
    on_road_links=False, boardings=True, alightings=True, transfers=True
)

# compute TC congested gtime
alpha = 0.15
beta = 4
sm.links["capacity"] = 0.1 
sm.links["time"] = sm.links["time"] * (1 + alpha * sm.links["load"] / sm.links["capacity"]) ** beta

# update pt_los with congested time
d = sm.links["time"].to_dict()
sm.pt_los['in_vehicle_time'] = sm.pt_los['link_path'].apply(lambda ls: sum([d[t] for t in ls]))
cols = [
    'access_time',
    'footpath_time',
    'waiting_time',
    'boarding_time',
    'in_vehicle_time',
]

sm.pt_los['time'] = sm.pt_los[cols].T.sum()

  0%|          | 0/1 [00:00<?, ?it/s]

volume: 100%|██████████| 1/1 [00:01<00:00,  1.86s/it]


# C. CAR ASSIGNMENT

In [ ]:
sm.road_links["capacity"] = 1 ### TODO Louison : à partir de vitesse max et lanes

In [ ]:
sm.zone_to_road["length"] = sm.zone_to_road.geometry.apply(lambda g: g.length)

In [ ]:
car = sm.copy()
CAR_OCCUPANCY_RATE = 1.5  # TODO Louison: valeur de l'occupation moyenne des voitures 
# CARS
car_los = sm.los.loc[sm.los['pathfinder_session']=='car_aon']
car_los['volume'] *= 1 / CAR_OCCUPANCY_RATE
car.volumes = car_los[["origin", "destination", "volume"]]

# car.step_road_pathfinder(method='aon', segments=["volume"])
car.step_road_pathfinder(method='msa', segments=["volume"], tolerance=10) # pour les itérations intermédiaires, pas besoin d’un relgap à 0.01%

# TODO Louison: remplacer roadpathfinder MSA par un simple AON + mise à jour des temps sur les road_links à partir des courbes BPR T0*(1+\a * (T/C)**\b)

# D. MERGE

In [ ]:
#TODO Louison: merge update car los, pt-los, with the rest (bike)
#TODO Louison: back to logit (boucle)

# TOZIPPEDPICKLES

In [ ]:
# TODO: lighten los

In [ ]:
sm.to_zippedpickles(
    mpath + '{scen}/assignment'.format(scen=SCENARIO), 
)

In [ ]:
end_of_notebook

# CONTROL

In [ ]:
temp = sm.los[sm.los['origin']==sm.los['destination']]
round(
    temp.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() /\
    temp[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() * 100
)

In [ ]:
round(
    sm.los.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() /\
    sm.los[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() * 100
)

In [ ]:
round(
    sm.los.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum() /\
    sm.los[['pt_captive', 'car_owners', 'moto_owners']].sum() * 100
)